# Extracting rules from the grammar text

In [1]:
import json
import nltk
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm



In [2]:
file = "manual_on_modern_kannada_classified.json"

with open(file,encoding="utf-8") as f:
    data = json.load(f)

data

[{'heading': 'A Manual of Modern Kannada',
  'text': 'Robert J. Zydenbos Bibliographic information published by the Deutsche Nationalbibliothek The Deutsche Nationalbibliothek lists this publication in the Deutsche Nationalbibliografie; detailed bibliographic data are available on the Internet at http://dnb.dnb.de. This book is published under the Creative Commons Attribution 4.0 License (CC-BY-NC-ND 4.0). The cover is subject to the Creative Commons License CC-BY-ND 4.0. Published by CrossAsia-eBooks, Heidelberg University Library 2020. The electronic open access version of this work is permanently available on the website of CrossAsia-eBooks: https://crossasia-books.ub.uni-heidelberg.de/xasia URN: urn:nbn:de:bsz:16-xabooks-736-9 DOI: https://doi.org/10.11588/xabooks.736 Text © 2020 by Robert J. Zydenbos Cover  illustration: Ferdinand Kittel, A Kannaḍa-English dictionary. Mangalore, 1894. Universitätsbibliothek Tübingen, shelf mark Ci XIV 80 a. https://doi.org/10.20345/digitue.12651 I

In [19]:
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')
tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
text = "Hello. My name is P.K.John and this is a test, i.e an example (e.g.) of tokenization. And another one."
tokenizer.tokenize(text)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Varun\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


['Hello.',
 'My name is P.K.John and this is a test, i.e an example (e.g.)',
 'of tokenization.',
 'And another one.']

In [20]:
import spacy
nlp = spacy.load("en_core_web_sm")

sentences = [sent.text.strip() for sent in nlp(text).sents]

sentences

['Hello.',
 'My name is P.K.John and this is a test, i.e an example (e.g.) of tokenization.',
 'And another one.']

In [21]:


sentences_data = []
filtered_out_data = []

# Tags to filter out with the threshold
tags_to_filter = {
    "Publishing information": 1.0,
    "Structure": 2.0,
    "Culture": 2.0,
    "History": 2.0,
    "Phonetics": 2.0
}

for section in data:
    sorted_tags = section["sorted_tags"]
    sorted_tag_names = [tag[0] for tag in sorted_tags]
    tag_word_counts = section["tag_word_counts"]
    tag_counts_heading  = section["tag_counts_heading"]
   
    if any(tag[0] in tags_to_filter and tag[1] >= tags_to_filter[tag[0]] for tag in sorted_tags):
        filtered_out_data.append(section)
        continue

    # Check if there are any other tags that are not in the tags_to_filter
    other_tags = [tag for tag in sorted_tag_names if tag not in tags_to_filter]
    if not other_tags:
        filtered_out_data.append(section)
        continue
    print(sorted_tag_names, tags_to_filter)
    text = section["text"]
    print(text,"\n")
    # Use spacy to split the text into sentences
    sentences = [sent.text.strip() for sent in nlp(text).sents]
    sentences_data.append({
        "text":text,
        "sentences":sentences,
        "sorted_tags":sorted_tags,
        #"sorted_tag_names":sorted_tag_names
        "tagged_words":tag_word_counts,
        "tagged_words_heading":tag_counts_heading
    })

len(sentences_data), len(data)
        

['Future', 'History', 'Present', 'Culture', 'Tense'] {'Publishing information': 1.0, 'Structure': 2.0, 'Culture': 2.0, 'History': 2.0, 'Phonetics': 2.0}
A new learner’s manual of Kannada for non-Indian learners is not pub- lished often. The reasons which persons may have for learning a lan- guage can differ widely, and the present author has tried to satisfy a variety of interests and wishes. The result, obviously, is a book that most probably also contains information that is of little interest for a certain specific individual reader or the other. It contains a bit of in- formation about earlier historical stages of the language, about general Dravidian linguistics, about social customs and how these are reflected in the language, about idioms, about colloquialisms and dialects; but all these topics cannot be treated in full detail in a single book. The author hopes that the book will serve as a solid and useful basis for the individual studies of each reader, in whatever direction t

(177, 312)

# Check filtered out data for sanity

In [22]:
filtered_out_data

[{'heading': 'A Manual of Modern Kannada',
  'text': 'Robert J. Zydenbos Bibliographic information published by the Deutsche Nationalbibliothek The Deutsche Nationalbibliothek lists this publication in the Deutsche Nationalbibliografie; detailed bibliographic data are available on the Internet at http://dnb.dnb.de. This book is published under the Creative Commons Attribution 4.0 License (CC-BY-NC-ND 4.0). The cover is subject to the Creative Commons License CC-BY-ND 4.0. Published by CrossAsia-eBooks, Heidelberg University Library 2020. The electronic open access version of this work is permanently available on the website of CrossAsia-eBooks: https://crossasia-books.ub.uni-heidelberg.de/xasia URN: urn:nbn:de:bsz:16-xabooks-736-9 DOI: https://doi.org/10.11588/xabooks.736 Text © 2020 by Robert J. Zydenbos Cover  illustration: Ferdinand Kittel, A Kannaḍa-English dictionary. Mangalore, 1894. Universitätsbibliothek Tübingen, shelf mark Ci XIV 80 a. https://doi.org/10.20345/digitue.12651 I

# Process sentence data

In [23]:
sentences_data



[{'text': 'A new learner’s manual of Kannada for non-Indian learners is not pub- lished often. The reasons which persons may have for learning a lan- guage can differ widely, and the present author has tried to satisfy a variety of interests and wishes. The result, obviously, is a book that most probably also contains information that is of little interest for a certain specific individual reader or the other. It contains a bit of in- formation about earlier historical stages of the language, about general Dravidian linguistics, about social customs and how these are reflected in the language, about idioms, about colloquialisms and dialects; but all these topics cannot be treated in full detail in a single book. The author hopes that the book will serve as a solid and useful basis for the individual studies of each reader, in whatever direction those studies may lead. The author wishes to thank his first teachers of Kannada: the late Prof. Kamil V. Zvelebil (Rijksuniversiteit Utrecht, 

In [24]:
with open("sentences_data.json","w",encoding="utf-8") as f:
    json.dump(sentences_data,f)

# Load tagged data

In [2]:
import json

with open("sentences_data_tagged.json", "r", encoding="utf-8") as f:
    tagged_data = json.load(f)

tagged_data


[{'sentence': 'A new learner’s manual of Kannada for non-Indian learners is not pub- lished often.',
  'rule_tag': False},
 {'sentence': 'The reasons which persons may have for learning a lan- guage can differ widely, and the present author has tried to satisfy a variety of interests and wishes.',
  'rule_tag': False},
 {'sentence': 'The result, obviously, is a book that most probably also contains information that is of little interest for a certain specific individual reader or the other.',
  'rule_tag': False},
 {'sentence': 'It contains a bit of in- formation about earlier historical stages of the language, about general Dravidian linguistics, about social customs and how these are reflected in the language, about idioms, about colloquialisms and dialects; but all these topics cannot be treated in full detail in a single book.',
  'rule_tag': False},
 {'sentence': 'The author hopes that the book will serve as a solid and useful basis for the individual studies of each reader, in wh

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Load sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract sentences and labels
sentences = [item['sentence'] for item in tagged_data]
labels = [1 if item['rule_tag'] else 0 for item in tagged_data]

# Create embeddings
embeddings = model.encode(sentences)


print("Embeddings shape: ",embeddings.shape)
print("Labels distribution: ",np.unique(labels, return_counts=True))


W0603 16:54:34.835000 6996 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Embeddings shape:  (314, 384)
Labels distribution:  (array([0, 1]), array([239,  75], dtype=int64))


In [5]:
# Cat boost
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.2, random_state=42
)

# Train cat boost model with regularization to prevent overfitting
# Calculate class weights to handle imbalance
class_weights = dict(zip(np.unique(labels), 
                        compute_class_weight('balanced', 
                                          classes=np.unique(labels),
                                          y=labels)))

print("Class weights: ",class_weights)

# CatBoost with class weights
clf_cat = CatBoostClassifier(
    n_estimators=50,  
    learning_rate=0.05,
    max_depth=3,
    l2_leaf_reg=5,
    random_seed=42,
    early_stopping_rounds=10,
    class_weights=class_weights,
    verbose=False
)

# Logistic Regression with class weights 
clf_log = LogisticRegression(
    penalty='l2',
    C=1.0, 
    solver='liblinear',
    class_weight='balanced'
)

clf = clf_cat


clf.fit(X_train, y_train)

train_score = clf.score(X_train, y_train)
test_score = clf.score(X_test, y_test)

print(f"Training accuracy: {train_score:.3f}")
print(f"Test accuracy: {test_score:.3f}")

Class weights:  {0: 0.6569037656903766, 1: 2.0933333333333333}
Training accuracy: 0.956
Test accuracy: 0.778


In [16]:
# Classify all sentences and check results
with open("sentences_data.json","r",encoding="utf-8") as f:
    sentences_data = json.load(f)

# Get embeddings for all sentences
all_sentences = [sentence for item in sentences_data for sentence in item['sentences']]

print("Embedding no of sentences: ",len(all_sentences))
all_embeddings = model.encode(all_sentences)

# Classify all sentences
all_predictions = clf.predict(all_embeddings)

rule_sentences = []
non_rule_sentences = []
final_data = []
# Add predictions to sentences_data
for i, sentence in enumerate(all_sentences):
    #sentences_data[i]['rule_tag'] = all_predictions[i]
    if all_predictions[i] == 1:
        rule_sentences.append(f"<{i}> - {sentence}" )
    else:
        non_rule_sentences.append(f"<{i}> - {sentence}" )
    final_data.append({
        "sentence":sentence,
        "rule_tag":True if all_predictions[i] == 1 else False
    })

print("No of rule sentences: ",len(rule_sentences))
print("No of non rule sentences: ",len(non_rule_sentences))







Embedding no of sentences:  2617
No of rule sentences:  427
No of non rule sentences:  2190


In [17]:
# Print
print("Rule sentences: ")
for i in range(len(rule_sentences)):
    print(rule_sentences[i])
    print("")
print("--------------------------------")
print("Non rule sentences: ")
for i in range(len(non_rule_sentences)):
    print(non_rule_sentences[i])
   


Rule sentences: 
<18> - The basic word order in Kannada is subject-object-verb (SOV).

<19> - Adver- bial expressions of time, place and mode generally do not appear at the end of a sentence.

<21> - Like most languages of the world, Kannada has no words corresponding to the English ‘the’ and ‘a’.

<22> - Definiteness or indefiniteness is usually clear from the context.

<24> - This means that every word carries a basic meaning, and this meaning is modified by means of suffixes.

<25> - 4  Thus the entire verb system is largely a matter of suffixa- tion, with suffixes that are added to verb roots to indicate tense (past, present, future), person, etc.

<27> - + d (past tense) + enu (first person singular), i.e., call-[past tense]-I = ‘I called’; ¢ೊೆೆನು hoḍedenu as strike-[past tense]-I ‘I struck’ (from ¢ೊೆ hoḍe ‘to strike’); ಬೆೆನು baredenu as ‘I wrote’ (from ಬೆ bare ‘to write’); ಉĪೆನು uḷidenu as ‘I remained’ (from ಉĪ uḷi ‘to remain’), etc.

<28> - By using a different final en

In [18]:
# Save updated sentences_data
with open("sentences_data_tagged_catboost.json", "w", encoding="utf-8") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)
    

# Save the model
clf.save_model("rule_extraction_model.cbm")



In [19]:

# Load active learning rule extraction "rule_extraction_model_iteration_4.cbm"
clf.load_model("rule_extraction_model_iteration_4.cbm")

In [20]:
# classify all sentences again
# Classify all sentences and check results
with open("sentences_data.json","r",encoding="utf-8") as f:
    sentences_data = json.load(f)

# Get embeddings for all sentences
all_sentences = [sentence for item in sentences_data for sentence in item['sentences']]

print("Embedding no of sentences: ",len(all_sentences))
all_embeddings = model.encode(all_sentences)

# Classify all sentences
all_predictions = clf.predict(all_embeddings)

rule_sentences = []
non_rule_sentences = []
final_data = []
# Add predictions to sentences_data
for i, sentence in enumerate(all_sentences):
    #sentences_data[i]['rule_tag'] = all_predictions[i]
    if all_predictions[i] == 1:
        rule_sentences.append(f"<{i}> - {sentence}" )
    else:
        non_rule_sentences.append(f"<{i}> - {sentence}" )
    final_data.append({
        "sentence":sentence,
        "rule_tag":True if all_predictions[i] == 1 else False
    })

print("No of rule sentences: ",len(rule_sentences))
print("No of non rule sentences: ",len(non_rule_sentences))







Embedding no of sentences:  2617
No of rule sentences:  581
No of non rule sentences:  2036


In [23]:
# Save rule sentences
with open("rule_sentences.csv","w",encoding="utf=8") as f:
    for sentence in rule_sentences:
        index,sentence = sentence.split("> - ")
        index = index.split("<")[1]
        index = int(index)
        f.write(f'"{index}","{sentence}"\n')



In [6]:
# Read data
import pandas as pd
with open("rule_sentences.csv","r",encoding="utf-8") as f:
    rule_sentences = f.readlines()

data = []
for sentence in rule_sentences:
    index = sentence.split(",")[0]
    sentence = sentence.split(",")[1:]
    index = int(index.strip('"'))
    sentence = ",".join(sentence).strip('"')
    data.append((index, sentence.strip('')))

df = pd.DataFrame(data, columns=["sentence_index", "sentence"])
# Remove the default index column
df = df.set_index("sentence_index")
df

,sentence
sentence_index,
13,"Although the languages of northern India, such..."
21,"Like most languages of the world, Kannada has ..."
23,The languages of the Dravidian family are of a...
24,This means that every word carries a basic mea...
25,4 Thus the entire verb system is largely a ma...
...,...
2585,Only if there is no reason for expressed respe...
2587,"As in English, the semantics of ಮನುಷ½ manuṣya ..."
2588,One could also say about a woman ಅವಳುಾನುಷ½ av...


In [7]:
# Send to GPT
import openai

with open("api_key.txt", "r") as f:
    openai_api_key = f.read().strip()

from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

response = client.responses.create(
  model="gpt-4o-mini",
  input="""You are an expert in computational linguistics.

Given a descriptive grammar sentence, extract the underlying grammatical rule and convert it into a formal rewrite rule suitable for implementation using Finite-State Transducers (FSTs). Use the following YAML format:

- rule_id: A short unique ID
  description: A plain-language summary of what the rule does
  condition: When or where the rule applies
  action: What transformation is made
  fst_rule: A formal rule in the style of X → Y / L ___ R
  example: An example transformation of an input string into an output string

  Now extract the rule from the following sentence:
  "In kannada, the verb typically comes at the end of the sentence, and the subject is often omitted if it is clear from context."
"""
)

print(response)




Response(id='resp_6852b79f5a70819d8f5fe89e614cfce400d21d4172968cda', created_at=1750251423.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_6852b79fd9ac819db376ddfea511b39800d21d4172968cda', content=[ResponseOutputText(annotations=[], text='Here’s the extracted rule in your specified YAML format:\n\n```yaml\n- rule_id: verb_position_end\n  description: Indicates that the verb usually appears at the end of a Kannada sentence.\n  condition: When a sentence is constructed in Kannada.\n  action: Reorder the sentence structure to place the verb at the end.\n  fst_rule: S → NP VP / ___ S\n  example: "He eats" → "He food eats"\n```  \n\nThis rule captures the information about the typical verb placement in Kannada sentences. The example transformation demonstrates a simple rearrangement, though actual Hindi or Kannada sentences would involve more complex structures. Let me know if y

In [8]:
response_text = response.output[0].content[0].text

print(response_text)

Here’s the extracted rule in your specified YAML format:

```yaml
- rule_id: verb_position_end
  description: Indicates that the verb usually appears at the end of a Kannada sentence.
  condition: When a sentence is constructed in Kannada.
  action: Reorder the sentence structure to place the verb at the end.
  fst_rule: S → NP VP / ___ S
  example: "He eats" → "He food eats"
```  

This rule captures the information about the typical verb placement in Kannada sentences. The example transformation demonstrates a simple rearrangement, though actual Hindi or Kannada sentences would involve more complex structures. Let me know if you'd like more information or additional rules!


In [9]:


base_prompt = f"""You are an expert in computational linguistics. You are given a descriptive grammar sentence in english, about the language "Kannada".
Given a descriptive grammar sentence in English, extract the underlying grammatical rule it describes in simple words, and convert it into a formal rewrite rule suitable for implementation using Finite-State Transducers (FSTs). 
If the rule cannot be expressed in a formal rewrite rule, leave the fst_rule field empty. 
Use the following YAML format:

  description: A simple plain-language summary of what the rule does
  condition: When or where the rule applies
  action: What transformation is made
  fst_rule: A formal rule in the style of X → Y / L ___ R

Here are some examples:

---

Sentence: "One must choose between ಇದು idu ‘this [thing]’ and ಅದು adu ‘that [thing]’."

  description: Kannada requires a proximal or distal demonstrative pronoun instead of a neutral 'it'
  condition: When referring to a third-person inanimate entity
  action: Insert either 'idu' or 'adu' based on proximity
  fst_rule: ε → idu | adu / ___ N[+INANIMATE]
  example: "book" → "idu book" (if near)

---

Sentence: "In Kannada, equative sentences do not use a copula in the present tense."

  description: The copula is omitted in equative present-tense constructions
  condition: When expressing identity in the present tense
  action: Delete the copula 'is'
  fst_rule: None
  example: None

---

Sentence: "The plural suffix in Kannada is ‘gaLu’, which is added to nouns to mark plurality."

  description: Kannada uses the suffix 'gaLu' to mark plural nouns
  condition: When a noun is marked as plural
  action: Add suffix 'gaLu' to the noun
  fst_rule: ε → gaLu / N[+PLURAL] ___ #
  example: "magu [NOUN, PLURAL]" → "magugaLu"

---

Sentence: "To express the past tense for 'to be', Kannada uses the form ‘iddanu’ for masculine singular subjects."

  description: Kannada uses the auxiliary verb form 'iddanu' to indicate past tense for masculine singular
  condition: When the subject is third-person masculine singular and the verb is 'to be' in past tense
  action: Replace null verb with 'iddanu'
  fst_rule: ε → iddanu / PRO[+3MSG] ___ [BE][+PAST]
  example: "avanu [BE][+PAST]" → "avanu iddanu"

---

Now extract the rule from the following sentence:

Sentence: "{{input_sentence}}"
"""



In [13]:
from tqdm import tqdm

gpt_responses = []

LIMIT = 100
for index, sentence in tqdm(df.iterrows()):
    LIMIT -= 1
    if LIMIT < 0:
        break
    print(f"Processing sentence {index}: {sentence}")
    response = client.responses.create(
        model="gpt-4o-mini",
        input=base_prompt.format(input_sentence=sentence)
    )
    print(response.output[0].content[0].text)
    gpt_responses.append({
        "sentence_index": index,
        "sentence": sentence["sentence"],
        "gpt_response": response.output[0].content[0].text
    })
    # Here you can save the response or process it further as needed

0it [00:00, ?it/s]

Processing sentence 13: sentence    Although the languages of northern India, such...
Name: 13, dtype: object


1it [00:01,  1.49s/it]

```yaml
description: The sentence discusses languages in northern India.
condition: When referring to the languages spoken in northern India
action: No specific transformation indicated
fst_rule: 
```
Processing sentence 21: sentence    Like most languages of the world, Kannada has ...
Name: 21, dtype: object


2it [00:02,  1.25s/it]

...

```yaml
description: Kannada uses a specific form to refer to neuter subjects in certain constructions.
condition: When the subject is third-person plural neuter
action: Use the form 'they are' specifically for neuter
fst_rule: ε → they are / PRO[+3PL][+NEUTRAL] ___ #
example: "They (neuter)" → "they are"
```
Processing sentence 345: sentence    Whether a finite form of iru is present or pas...
Name: 345, dtype: object


99it [03:46,  2.02s/it]

```yaml
description: Kannada requires a finite form of the verb 'iru' to indicate presence or past tense.
condition: When expressing existence in the present or past tense
action: Insert a finite form of 'iru' as required
fst_rule: ε → iru / ___ [EXISTENCE][+PRES|+PAST]
example: None
```
Processing sentence 346: sentence    person (gender) singular (gender) plural 1 enu...
Name: 346, dtype: object


100it [03:49,  2.29s/it]

```yaml
description: The sentence structure in Kannada distinguishes between singular and plural forms based on the person and gender.
condition: When marking nouns or pronouns as singular or plural based on person and gender.
action: Attach appropriate markers for singular or plural forms based on given gender and person.
fst_rule: ε → g / N[+SINGULAR] ___ # | ε → ga / N[+PLURAL] ___ #
example: "avanu [NOUN, SINGULAR]" → "avanu enu", "avaru [NOUN, PLURAL]" → "avaru ga"
```


In [11]:
# Store the responses in a text file for now
with open("gpt_responses.txt", "w", encoding="utf-8") as f:
    for item in gpt_responses:
        f.write(f"Sentence Index: {item['sentence_index']}\n")
        f.write(f"Sentence: {item['sentence']}\n")
        f.write(f"GPT Response: {item['gpt_response']}\n")
        f.write("\n---\n\n")